# 06 · Flows: Explicit, Stateful Control Flow

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama for §6 only; the rest runs offline.

Crews are good at *doing* work; they're not good at *deciding the order of work*
in a way you can guarantee. **Flows** are CrewAI's answer: a Python class whose
methods are steps, wired together with decorators (`@start`, `@listen`,
`@router`), sharing a typed **state**. Flows give you branching, joins, loops
and (in later notebooks) persistence and human feedback, and any step can run a
crew, a single agent, or plain code.

**You will learn**
- The anatomy of a Flow: `@start`, `@listen`, return values and **state**
- **Routers** for branching, and a naming rule that prevents infinite loops
- **Joins** with `and_` / `or_`, and **loops** with a guard (`max_method_calls`)
- Running **crews, agents and plain LLM calls** inside Flow steps
- When to use a Flow, a Crew, or both

**Prerequisites:** notebooks 01–05. *Optional background:* `00_CONCEPTS/02` builds a
graph engine by hand; Flows are CrewAI's version of it.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

from pydantic import BaseModel
from crewai.flow import Flow, start, listen, router, and_, or_

## 1 · Why Flows

A crew decides *within* its process how work gets done. Real applications also
need guarantees *around* that work: "always check the draft before saving", "if
the classifier says billing, go here", "retry at most three times", "stop and ask
a human". Those are **control flow**, and a Flow makes it explicit:

```
                 ┌──► handle_billing ──┐
 @start ─► @router(classify)            ├──► @listen(and_/or_) ─► …
 classify        └──► handle_tech    ──┘
                       ▲  each step: plain Python, an LLM call, an agent, or a whole crew
         self.state ── shared, typed, (optionally) persisted
```

| | Crew | Flow |
|---|---|---|
| Who decides the order | the process (and the manager, if hierarchical) | **you**, via decorators |
| State | task outputs passed as context | a typed `self.state` every step can read and write |
| Branching / loops | not really | routers, joins, loops |
| Persistence, human feedback | limited | `@persist`, `@human_feedback` (notebooks 08–09) |

## 2 · Anatomy of a Flow

A Flow is a class. Methods marked `@start()` run first. `@listen(other)` runs
when `other` finishes, and receives its return value. Everything can read and
write `self.state`, here a Pydantic model, which also gets an automatic `id`.

In [ ]:
class TicketState(BaseModel):
    text: str = ''
    category: str = ''
    log: list[str] = []


class Hello(Flow[TicketState]):
    @start()
    def receive(self):
        self.state.log.append('received')
        return self.state.text.upper()                 # return values go to listeners

    @listen(receive)
    def acknowledge(self, shouted):                    # ← receives receive()'s return value
        self.state.log.append('acknowledged')
        return f'We got your message: {shouted}'


flow = Hello(suppress_flow_events=True)                # suppress = don't print a panel per step
result = await flow.kickoff_async(inputs={'text': 'my robot is late'})
print('result:', result)
print('state :', flow.state)

- **`inputs=`** fills matching state fields before the flow starts.
- **The final result** is the return value of the last method that ran.
- **`suppress_flow_events=True`** stops CrewAI printing a status panel for every
  step. Leave it off while debugging, since the panels show what ran.
- Flows run fine with `await flow.kickoff_async()` in notebooks (and with
  `flow.kickoff()` in scripts).

## 3 · Routers: branching

A `@router(...)` method returns a **label** (a string). Methods that
`@listen('that label')` run next:

In [ ]:
class Triage(Flow[TicketState]):
    @start()
    def classify(self):
        t = self.state.text.lower()
        self.state.category = 'billing' if any(w in t for w in ('charge', 'refund', 'invoice')) else 'technical'

    @router(classify)
    def route(self):
        return self.state.category                      # the label decides the branch

    @listen('billing')
    def handle_billing(self):
        return 'refund desk'

    @listen('technical')
    def handle_technical(self):
        return 'support engineer'


for text in ('I was charged twice', 'The app crashes on login'):
    f = Triage(suppress_flow_events=True)
    print(f'{text:<28} → {await f.kickoff_async(inputs={"text": text})}')

**A naming rule that protects you.** A router label must not be the *name* of the
method that listens to it. If `route` returned `'billing'` and the listener were
also *called* `billing`, the method would be triggered by its own completion,
an infinite loop. CrewAI rejects that flow definition at creation time:

In [ ]:
class Broken(Flow[TicketState]):
    @start()
    def route_it(self):
        return 'x'

    @router(route_it)
    def decide(self):
        return 'billing'

    @listen('billing')
    def billing(self):                                  # label == method name
        return 'never'

try:
    Broken()
except Exception as err:
    print(type(err).__name__, '→', str(err).split('[type=')[0][-240:])

That's why the branches above are called `handle_billing` / `handle_technical`.

## 4 · Joins: `and_` and `or_`

Two steps can start from the same trigger and run **concurrently**. A later step
decides how to join them:

- `@listen(and_(a, b))`: run after **both** finished (fan-in).
- `@listen(or_(a, b))`: run after **either** finished (first one wins).

In [ ]:
import asyncio, time

class Research(BaseModel):
    notes: list[str] = []
    report: str = ''


class Parallel(Flow[Research]):
    @start()
    def begin(self):
        return 'robots'

    @listen(begin)
    async def legal(self, topic):
        await asyncio.sleep(1)                         # pretend: a slow lookup
        self.state.notes.append(f'legal notes on {topic}')

    @listen(begin)
    async def technical(self, topic):
        await asyncio.sleep(1)
        self.state.notes.append(f'technical notes on {topic}')

    @listen(and_(legal, technical))                    # waits for BOTH
    def write(self):
        self.state.report = ' | '.join(sorted(self.state.notes))
        return self.state.report


t0 = time.perf_counter()
f = Parallel(suppress_flow_events=True)
print(await f.kickoff_async(), f'({time.perf_counter() - t0:.1f}s: the two 1-second steps overlapped)')

Flow methods can be **`async`**, and concurrent steps really overlap: 1 second,
not 2. Both appended to `self.state.notes`. That's safe here because list
appends don't conflict, but unlike LangGraph there are no reducers: parallel
steps that *overwrite* the same field will simply race. Give parallel steps
separate fields, or append.

## 5 · Loops, and the guard

A loop is a router that sends control back to a label an earlier step listens
to. `@listen(or_(begin, 'revise'))` means "run on the first pass, and again
whenever a router says `revise`":

In [ ]:
class Draft(BaseModel):
    text: str = ''
    rounds: int = 0
    status: str = ''

MAX_ROUNDS = 3

class Refine(Flow[Draft]):
    @start()
    def begin(self):
        return 'go'

    @listen(or_(begin, 'revise'))
    def write(self):
        self.state.rounds += 1
        self.state.text = f'draft v{self.state.rounds}'

    @router(write)
    def check(self):
        good_enough = self.state.rounds >= 2           # stand-in for a real quality check
        if good_enough or self.state.rounds >= MAX_ROUNDS:
            return 'done'
        return 'revise'

    @listen('done')
    def finish(self):
        self.state.status = f'finished after {self.state.rounds} rounds'
        return self.state.status


f = Refine(suppress_flow_events=True)
print(await f.kickoff_async(), '|', f.state.text)

The loop has its own guard (`MAX_ROUNDS` in state, checked by the router). As a
last resort, a Flow also refuses to call any single method more than
**`max_method_calls`** times per run (default **100**) and raises a
`RecursionError`, so a loop without a proper guard can't spin forever:

In [ ]:
class Runaway(Flow[Draft]):
    @start()
    def begin(self):
        return 'go'

    @listen(or_(begin, 'again'))
    def spin(self):
        self.state.rounds += 1

    @router(spin)
    def always_again(self):
        return 'again'                                 # no exit condition!

f = Runaway(suppress_flow_events=True, max_method_calls=10)
try:
    await f.kickoff_async()
except RecursionError as err:
    print('stopped after', f.state.rounds, 'rounds:', str(err)[:110], '…')

Like `recursion_limit` in LangGraph or `max_turns` in the Agents SDK, it's a crash,
not an answer. Always write your own exit condition; keep `max_method_calls`
as the safety net.

## 6 · Crews, agents and LLM calls inside a Flow

Each Flow step can do its work in the most appropriate way:

| Inside a step | Use when |
|---|---|
| plain Python | the decision or transformation is deterministic |
| `llm.call(prompt)` | one quick model call, no tools |
| `await agent.kickoff_async(prompt)` | one agent with tools, no crew around it |
| `await crew.kickoff_async(inputs=...)` | a multi-agent piece of work |

A support flow: a cheap LLM call classifies, code routes, and a small crew
handles the hard branch:

In [ ]:
from crewai import Agent, Task, Crew

llm = get_llm()


class Support(Flow[TicketState]):
    @start()
    def classify(self):
        answer = llm.call(f"Reply with one word, billing or technical: {self.state.text}").strip().lower()
        self.state.category = answer if answer in ('billing', 'technical') else 'technical'  # code decides the fallback

    @router(classify)
    def route(self):
        return self.state.category

    @listen('billing')
    def handle_billing(self):
        return 'Billing: refund request logged.'

    @listen('technical')
    async def handle_technical(self):
        engineer = Agent(role='Support engineer', goal='Give one precise troubleshooting step',
                         backstory='Practical and brief.', llm=llm)
        crew = Crew(agents=[engineer], tasks=[Task(description=f'Customer problem: {self.state.text}',
                                                   expected_output='One troubleshooting step', agent=engineer)])
        return (await crew.kickoff_async()).raw


for text in ('I was charged twice for my robot kit', 'The robot stops at every curb'):
    f = Support(suppress_flow_events=True)
    print(f'{text}\n  → [{(await f.kickoff_async(inputs={"text": text}))[:150]}]\n')

This combination is the heart of CrewAI design: **the Flow guarantees the
structure; crews and agents do the judgement work inside it.** It's the same
"workflow with agents inside" shape as in the other sections, expressed with
decorators.

**Visualising a flow.** `flow.plot('my_flow')` writes an interactive HTML
diagram of the steps and their connections. It's a useful check that you wired
what you meant.

## 7 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Router label equals a listener's name | flow rejected at creation | name handlers `handle_<label>` |
| Loop without an exit condition | `RecursionError` at `max_method_calls` | a counter in state, checked by the router |
| Parallel steps overwriting one field | lost updates | separate fields, or append |
| `crew.kickoff()` inside an async step | runtime error in a running loop | `await crew.kickoff_async()` |
| Everything in one giant crew | no guarantees about order or checks | a Flow around smaller crews |

## 8 · Check yourself

<details><summary><b>1.</b> What do <code>@start</code>, <code>@listen</code> and <code>@router</code> do?</summary>

`@start` marks entry steps. `@listen(x)` runs a method after `x` (a method or a router label) completes, receiving its return value. `@router(x)` runs after `x` and returns a label that selects which listeners run next.
</details>

<details><summary><b>2.</b> Why does CrewAI reject a flow where a router returns <code>'billing'</code> and a method called <code>billing</code> listens for it?</summary>

The listener would be triggered by its own name, and so by its own completion: an infinite loop. CrewAI detects it when the flow is created.
</details>

<details><summary><b>3.</b> <code>and_</code> vs <code>or_</code>?</summary>

`and_(a, b)` runs the listener after both finished (a join). `or_(a, b)` runs it after either one (also used for loops: `or_(begin, 'revise')`).
</details>

<details><summary><b>4.</b> What happens if a loop never exits?</summary>

After `max_method_calls` calls to one method (default 100) the flow raises `RecursionError`. It's a safety net, so write an explicit exit condition in the router.
</details>

<details><summary><b>5.</b> In a Flow step, when would you use <code>llm.call</code> vs a crew?</summary>

`llm.call` for a single, tool-free model call (classify, summarise). A crew when the work needs agents with tools, several roles or collaboration.
</details>

## Takeaway

A **Flow** is a class of steps wired by `@start`, `@listen` and `@router`, sharing
typed **state**. It gives you branching, joins (`and_`/`or_`) and guarded loops,
and each step can be plain code, an LLM call, an agent or a crew. Use Flows for
structure and guarantees, and crews for the work inside them.

Next → `07_guardrails.ipynb`: validating task outputs and retrying automatically.